# Applied z-push versus the phase of the controlled trajectory

This notebook reads the **saved successful** sign-invariant, z-observation/z-control CSE result for positive event 0. It does not rerun OSSE, DA, or CSE and does not change any intervention. For each applied $u_z[i]$, it compares $\operatorname{sign}(u_z[i])$ with $\operatorname{sign}(q[i+L])$, where $q=x\dot x=\sigma x(y-x)$ is evaluated along the **already controlled** nature trajectory and $L=-20,\ldots,20$ model steps. Positive $L$ means after the intervention.

To compare percentages fairly, the figure uses the same interventions at every lag and excludes pushes lacking the full lag window near the trajectory boundaries. A separate lag-zero check uses **all** applied pushes and should reproduce the previously reported ~78% agreement. Future controlled states include the effects of the current and subsequent pushes; this is a descriptive alignment test, not a causal no-push forecast or information available to the controller.


In [ ]:
import pickle

from cse_controlled_phase_lag_diagnostic import (
    summarize_controlled_phase_lags, plot_controlled_phase_lags,
)
from project_paths import CSE_ROOT

EXPERIMENT_ROOT = CSE_ROOT / 'z_control_z_obs_sign_invariant_qiwenevents'
RESULT_PATH = (
    EXPERIMENT_ROOT / 'results' / 'initial_nature_positive'
    / 'cse_z_control_z_obs_sign_invariant_event_0_D0p05_qiwenevents.pkl'
)
if not RESULT_PATH.is_file():
    raise FileNotFoundError(RESULT_PATH)
with RESULT_PATH.open('rb') as stream:
    saved_cse = pickle.load(stream)
assert saved_cse['success'] is True
assert saved_cse['regime_reference'] == 'sign_invariant'
assert saved_cse['control_indices'] == [2]
print('Saved result:', RESULT_PATH)
print('Completed cycles:', saved_cse['n_completed'])
print('Model step dt:', saved_cse['cse_summary']['fcst_dt'])


In [ ]:
LAGS = range(-20, 21)
lag_rows, lag_metadata = summarize_controlled_phase_lags(
    saved_cse, lags=LAGS, sigma=10.0
)
print('All-push lag-zero consistency check: '
      f"{lag_metadata['lag0_all_agree']}/"
      f"{lag_metadata['lag0_all_defined']} = "
      f"{lag_metadata['lag0_all_percent']:.2f}%")
assert abs(lag_metadata['lag0_all_percent'] - 78.34) < 1.0
print(f"Common lag-window sample: {lag_metadata['n_common_pushes']} "
      f"of {lag_metadata['n_all_pushes']} pushes; "
      f"{lag_metadata['n_excluded_boundaries']} excluded at boundaries.")
print('Lag   Time    Defined  Zero q   Agree  Agreement')
for row in lag_rows:
    print(f"{row['lag']:+3d}  {row['lag_time']:+.3f}  "
          f"{row['n_defined']:>7}  {row['n_zero_q']:>6}  "
          f"{row['n_agree']:>6}  {row['agreement_percent']:>8.2f}%")


In [ ]:
from IPython.display import Image, display

figure_path = (
    EXPERIMENT_ROOT / 'figures' / 'initial_nature_positive'
    / 'uz_sign_vs_controlled_q_lag_event_0_D0p05.png'
)
figure_path = plot_controlled_phase_lags(
    lag_rows, lag_metadata, figure_path
)
best = max(lag_rows, key=lambda row: row['agreement_percent'])
lag_zero = next(row for row in lag_rows if row['lag'] == 0)
print(f"Plotted lag-zero agreement: {lag_zero['agreement_percent']:.2f}%")
print(f"Best lag: {best['lag']:+d} model steps; "
      f"tau = {best['lag_time']:+.3f}; "
      f"maximum agreement = {best['agreement_percent']:.2f}%")
print('New figure:', figure_path)
display(Image(filename=str(figure_path)))
